---
# Часть 2. Лабораторная работа «Т-тесты» (анализ датасета Психология)

**Цель работы:** оценить наличие статистически значимых гендерных различий по психологическим и интеллектуальным показателям учащихся с помощью двухвыборочного t-критерия Стьюдента.

**Задачи:**
1. Загрузить набор данных `Психология.xlsx` в DataFrame и провести первичную проверку структуры и качества данных (пропуски, дубликаты, названия столбцов).
2. Сформулировать статистические гипотезы ($H_0$ и $H_1$), обосновать выбор Т-теста и его параметров (проверка равенства дисперсий).
3. Рассчитать t-статистику и p-value для каждого показателя и сделать статистические выводы.
4. Определить показатели, в которых гендерные различия проявились сильнее и слабее всего.

---
### Этап 1. Загрузка данных и первичная проверка
Загрузим файл `Психология.xlsx` и посмотрим на структуру таблицы, список столбцов, типы данных и наличие возможных ошибок (пропусков или полных дубликатов).


In [6]:
import pandas as pd
import numpy as np
from scipy import stats

# Загружаем датасет Психология.xlsx
df_psy = pd.read_excel('Психология.xlsx')

# Уберем лишние пробелы по краям в названиях столбцов, если они есть
df_psy.columns = df_psy.columns.str.strip()

print("Первые 5 строк таблицы:")
display(df_psy.head())

print("\nИнформация о типах данных и пропусках:")
df_psy.info()

print("\nКоличество полных дубликатов строк:", df_psy.duplicated().sum())
print("Распределение учащихся по полу:")
print(df_psy['пол'].value_counts())


Первые 5 строк таблицы:


,учащиеся,пол,класс,профиль ВУЗа,осведомленность,скрытые фигуры,пропущенные слова,арифметика,понятливость,исключение изображений,аналогии,числовые ряды,умозаключения,геометрическое сложение,заучивание слов,средний IQ,экстраверсия- интроверсия,нейротизм,средняя отметка
0,1,ж,гум,0,12,9,11,8,8,11,13,8,12,10,11,103,15,7,3.93
1,2,ж,гум,1,10,12,12,11,10,12,9,12,8,11,11,107,13,17,4.27
2,3,м,мат,1,11,8,9,11,11,12,9,11,9,11,8,100,10,19,3.87
3,4,ж,гум,0,14,12,14,13,13,9,9,9,12,11,12,116,13,11,4.57
4,5,ж,гум,0,12,12,9,10,9,12,5,10,3,11,9,93,16,8,4.14



Информация о типах данных и пропусках:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   учащиеся                   46 non-null     int64  
 1   пол                        46 non-null     object 
 2   класс                      46 non-null     object 
 3   профиль ВУЗа               46 non-null     int64  
 4   осведомленность            46 non-null     int64  
 5   скрытые фигуры             46 non-null     int64  
 6   пропущенные слова          46 non-null     int64  
 7   арифметика                 46 non-null     int64  
 8   понятливость               46 non-null     int64  
 9   исключение  изображений    46 non-null     int64  
 10  аналогии                   46 non-null     int64  
 11  числовые ряды              46 non-null     int64  
 12  умозаключения              46 non-null     int64  
 13  геометрическ

**Вывод по Этапу 1:**
Датасет `Психология.xlsx` успешно загружен. Таблица содержит данные по **46 учащимся** и **19 столбцам**:
* `учащиеся` — порядковый номер ученика (ID, не является психологическим показателем);
* `пол` и `класс` — текстовые категориальные признаки (`object`). Выборка идеально сбалансирована по полу: **23 девушки (`ж`)** и **23 юноши (`м`)**;
* `профиль ВУЗа` — бинарный признак выбора профиля (`0` или `1`);
* **15 количественных показателей тестирования:** 11 интеллектуальных субтестов (`осведомленность`, `скрытые фигуры`, `пропущенные слова`, `арифметика`, `понятливость`, `исключение изображений`, `аналогии`, `числовые ряды`, `умозаключения`, `геометрическое сложение`, `заучивание слов`), итоговый `средний IQ`, личностные шкалы (`экстраверсия- интроверсия`, `нейротизм`) и успеваемость (`средняя отметка`).

Пропущенных значений (`NaN`) и дубликатов строк в наборе данных не обнаружено (0 шт.), все числовые показатели имеют корректные типы данных (`int64` и `float64`).


### Этап 2. Оценка гендерных различий по показателям (Т-тест)

**а) Формулировка гипотез (уровень значимости α = 0.05):**
* **H0 (нулевая гипотеза):** средние значения показателя у юношей и девушек равны (гендерных различий нет).
* **H1 (альтернативная гипотеза):** средние значения у юношей и девушек значимо различаются (гендерные различия есть).

Если p-value < 0.05, то нулевая гипотеза отвергается (различия есть). Если p-value >= 0.05, то различия статистически не значимы.

**b) Выбор Т-теста и параметров:**
Юноши и девушки — это две независимые выборки (по 23 человека в каждой), поэтому выбираем двухвыборочный t-критерий для независимых выборок `stats.ttest_ind()`. Альтернатива двусторонняя (`alternative='two-sided'`). Равенство дисперсий проверим тестом Левена (`stats.levene`): так как везде p-value > 0.05, дисперсии равны, поэтому задаем параметр `equal_var=True`.


In [7]:
# Выбираем все показатели для анализа (все столбцы, кроме номера ученика, пола и текстового класса)
cols = df_psy.columns[3:]

# Разделяем выборку на две независимые группы: юноши (м) и девушки (ж)
boys_psy = df_psy[df_psy['пол'] == 'м'][cols]
girls_psy = df_psy[df_psy['пол'] == 'ж'][cols]

# Проверяем равенство дисперсий тестом Левена сразу по всем столбцам
levene_stat, levene_p = stats.levene(boys_psy, girls_psy)

# Проводим двухвыборочный независимый Т-тест сразу для всех показателей
t_stat, p_val = stats.ttest_ind(boys_psy, girls_psy, equal_var=True, alternative='two-sided')

# Собираем наглядную таблицу с результатами (статистика + p-значение + вывод)
results_psy = pd.DataFrame(index=cols)
results_psy['Среднее (м)'] = boys_psy.mean().round(2)
results_psy['Среднее (ж)'] = girls_psy.mean().round(2)
results_psy['Levene p-val'] = np.round(levene_p, 4)
results_psy['t-статистика'] = np.round(t_stat, 4)
results_psy['|t| (сила различий)'] = np.abs(t_stat).round(4)
results_psy['p-значение (p-value)'] = np.round(p_val, 4)
results_psy['Вывод (при α=0.05)'] = np.where(p_val < 0.05, 'Различия ЕСТЬ (значимы)', 'Различий НЕТ')

print("Результаты проверки гендерных различий по всем показателям:")
display(results_psy)

print("\nСортировка показателей по силе гендерных различий (от самых сильных к самым слабым):")
display(results_psy[['Среднее (м)', 'Среднее (ж)', 't-статистика', 'p-значение (p-value)', 'Вывод (при α=0.05)']].sort_values(by='p-значение (p-value)'))


Результаты проверки гендерных различий по всем показателям:


,Среднее (м),Среднее (ж),Levene p-val,t-статистика,|t| (сила различий),p-значение (p-value),Вывод (при α=0.05)
профиль ВУЗа,0.78,0.22,1.0000,4.5448,4.5448,0.0000,Различия ЕСТЬ (значимы)
осведомленность,10.30,11.26,0.4020,-1.4732,1.4732,0.1478,Различий НЕТ
скрытые фигуры,9.04,10.61,0.6176,-2.0113,2.0113,0.0504,Различий НЕТ
пропущенные слова,9.91,10.39,0.8566,-0.6218,0.6218,0.5373,Различий НЕТ
арифметика,11.70,10.70,0.5270,1.6686,1.6686,0.1023,Различий НЕТ
понятливость,9.91,10.22,0.8892,-0.5255,0.5255,0.6019,Различий НЕТ
исключение изображений,9.30,10.26,0.1636,-1.3573,1.3573,0.1816,Различий НЕТ
аналогии,11.30,10.52,0.1336,0.9194,0.9194,0.3629,Различий НЕТ
числовые ряды,10.43,9.35,0.5101,1.6023,1.6023,0.1163,Различий НЕТ
умозаключения,12.17,11.91,0.9458,0.3022,0.3022,0.7639,Различий НЕТ



Сортировка показателей по силе гендерных различий (от самых сильных к самым слабым):


,Среднее (м),Среднее (ж),t-статистика,p-значение (p-value),Вывод (при α=0.05)
профиль ВУЗа,0.78,0.22,4.5448,0.0000,Различия ЕСТЬ (значимы)
экстраверсия- интроверсия,11.57,14.61,-3.4441,0.0013,Различия ЕСТЬ (значимы)
скрытые фигуры,9.04,10.61,-2.0113,0.0504,Различий НЕТ
арифметика,11.70,10.70,1.6686,0.1023,Различий НЕТ
нейротизм,12.83,11.00,1.6145,0.1136,Различий НЕТ
числовые ряды,10.43,9.35,1.6023,0.1163,Различий НЕТ
заучивание слов,9.17,10.17,-1.5355,0.1318,Различий НЕТ
средняя отметка,3.98,4.15,-1.4897,0.1434,Различий НЕТ
осведомленность,10.30,11.26,-1.4732,0.1478,Различий НЕТ
исключение изображений,9.30,10.26,-1.3573,0.1816,Различий НЕТ


### Выводы по работе

1. **Проверка дисперсий:** Тест Левена для всех показателей дал p-value > 0.05 (от 0.1336 до 1.0), значит дисперсии в группах юношей и девушек равны и параметр `equal_var=True` выбран верно.
2. **Где ЕСТЬ гендерные различия (p-value < 0.05):**
   * **Профиль ВУЗа** (t = 4.5448, p = 0.0000): юноши намного чаще выбирают профиль 1 (среднее 0.78 против 0.22 у девушек).
   * **Экстраверсия-интроверсия** (t = -3.4441, p = 0.0013): у девушек показатель экстраверсии статистически значимо выше, чем у юношей (14.61 против 11.57).
   * **Скрытые фигуры** (t = -2.0113, p = 0.0504): формально p-value чуть больше 0.05 (различий нет), но значение находится прямо на границе значимости (у девушек средний балл выше: 10.61 против 9.04).
3. **Где гендерных различий НЕТ (p-value >= 0.05):**
   По всем остальным интеллектуальным субтестам (`осведомленность`, `пропущенные слова`, `арифметика`, `понятливость`, `исключение изображений`, `аналогии`, `числовые ряды`, `умозаключения`, `геометрическое сложение`, `заучивание слов`), а также по `нейротизму`, `среднему IQ` и `средней отметке` значимых различий между полами нет.
4. **Ответ на вопрос №3 (где различия сильнее и слабее всего):**
   * **Сильнее всего** различия проявились в показателях **профиль ВУЗа** (|t| = 4.5448, p = 0.0000) и **экстраверсия-интроверсия** (|t| = 3.4441, p = 0.0013), а среди субтестов IQ — в тесте **скрытые фигуры** (|t| = 2.0113, p = 0.0504).
   * **Слабее всего** различия проявились в **среднем IQ** (|t| = 0.1657, p = 0.8691 — средние почти одинаковые: 104.52 и 105.22) и в субтестах **умозаключения** (|t| = 0.3022, p = 0.7639) и **понятливость** (|t| = 0.5255, p = 0.6019).

